# Quantum CQ - IBM Real Smoke Test

Este notebook submete jobs reais quando `RUN_REAL_IBM = True`. Pode entrar em fila, demorar e consumir quota ou creditos. Nunca compartilhe nem commite seu token.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from quantum_cq import CQ

RUN_REAL_IBM = True

IBM_QUANTUM_TOKEN = "COLE_SEU_TOKEN_AQUI".strip()

IBM_QUANTUM_CHANNEL = "ibm_quantum_platform"
# None deixa a IBM/Qiskit Runtime resolver a instancia na hora da execucao.
# Se auto-discovery falhar, teste o outro channel antes de preencher um CRN/nome real.
IBM_QUANTUM_INSTANCE = None
IBM_QUANTUM_BACKEND = "least_busy"
IBM_QUANTUM_SHOTS = 32
IBM_QUANTUM_TIMEOUT = 300

In [ ]:
CQ.notebook_logs("INFO")

In [ ]:
if RUN_REAL_IBM:
    if not IBM_QUANTUM_TOKEN or IBM_QUANTUM_TOKEN == "COLE_SEU_TOKEN_AQUI":
        raise RuntimeError(
            "RUN_REAL_IBM=True, mas IBM_QUANTUM_TOKEN ainda esta como placeholder. "
            "Cole seu token IBM real na variavel IBM_QUANTUM_TOKEN antes de executar."
        )

In [ ]:
ibm = CQ.ibm(
    token=IBM_QUANTUM_TOKEN,
    channel=IBM_QUANTUM_CHANNEL,
    instance=IBM_QUANTUM_INSTANCE,
)

print(ibm)
print(ibm.safe_summary())

assert IBM_QUANTUM_TOKEN not in repr(ibm)
assert IBM_QUANTUM_TOKEN not in str(ibm.safe_summary())

In [ ]:
circuit = CQ.deutsch(case=2)
CQ.show(circuit)

In [ ]:
if RUN_REAL_IBM:
    result_real = CQ.run(
        circuit,
        mode="real",
        ibm=ibm,
        backend=IBM_QUANTUM_BACKEND,
        shots=IBM_QUANTUM_SHOTS,
        timeout=IBM_QUANTUM_TIMEOUT,
        title="Deutsch case=2 - IBM real smoke test",
    )

    result_real.show()
    print(result_real.global_metrics())
    display(result_real.to_dataframe())

    exp = result_real.experiments[0]
    print("backend_name:", exp.backend_name)
    print("job_id:", exp.job_id)
    print("status:", exp.status)
    print("counts:", exp.counts)
    print("classification:", result_real.classify("deutsch"))

    assert exp.backend_name
    assert exp.job_id
    assert exp.counts
    assert sum(exp.counts.values()) > 0
else:
    print("RUN_REAL_IBM=False")

In [ ]:
if RUN_REAL_IBM:
    result_multi_real = CQ.run(
        circuits=[CQ.deutsch(case=1), CQ.deutsch(case=2)],
        modes=["real"],
        ibm=ibm,
        backend=IBM_QUANTUM_BACKEND,
        shots=IBM_QUANTUM_SHOTS,
        timeout=IBM_QUANTUM_TIMEOUT,
        title="Deutsch case=1 e case=2 - IBM real multi-circuit smoke test",
    )

    result_multi_real.show()
    print(result_multi_real.global_metrics())
    display(result_multi_real.to_dataframe())

    assert len(result_multi_real.experiments) == 2
    assert all(exp.backend_name for exp in result_multi_real.experiments)
    assert all(exp.job_id for exp in result_multi_real.experiments)
    assert all(exp.counts for exp in result_multi_real.experiments)

In [ ]:
print("OK: IBM config segura.")
print("OK: Deutsch circuit criado.")
print("OK: IBM real smoke executado.")
print("OK: backend_name/job_id/status/counts coletados.")
print("OK: quantum_cq_ibm_real_smoke.ipynb executou com IBM real.")